# ChibiCreate — BENCHMARK COMPARATIVO: WAI-illustrious-SDXL

> ## BENCHMARK COMPARATIVO — NAO E PIPELINE OFICIAL
>
> Responde **uma** pergunta: *o WAI-illustrious-SDXL preserva a roupa e o
> design original melhor que o FLUX.2 klein 4B?*
>
> Nao altera o benchmark FLUX, as Runs 001/002/003 do FLUX, o Flow 01, os
> quality gates nem o design transfer.

---

## Multi-referencia via IP-Adapter

O checkpoint SDXL nao tem mecanismo proprio de referencia como o
`ReferenceLatent` do FLUX. Isso **nao** quer dizer que ele nao consiga usar
varias referencias: o **IP-Adapter** fornece multi-referencia real.

```
full_body ──► IPAdapterEncoder (peso 1.0) ──┐
face      ──► IPAdapterEncoder (peso 0.6) ──┼──► IPAdapterCombineEmbeds
outfit    ──► IPAdapterEncoder (peso 0.8) ──┘             │
                                                  IPAdapterEmbeds
                                                          │
                                                      KSampler
```

Encoder + Combine em vez de `IPAdapterAdvanced` empilhado em serie, porque
so assim o peso de **cada** referencia fica explicito e auditavel.

| run | referencias | workflow |
|---|---|---|
| 001 | `full_body` | `v1` |
| 002 | `full_body` — repeticao exata da 001 | `v1` |
| 003 | `full_body` + `face` + `outfit` | `v2` |

### Sobre a comparacao com o FLUX

- **FLUX** = multi-referencia pelo mecanismo proprio (`ReferenceLatent`)
- **WAI** = multi-referencia por **IP-Adapter**

Sao implementacoes **diferentes**. A comparacao e sobre o **resultado visual
com as mesmas referencias de entrada**, nunca sobre equivalencia de
arquitetura.

### Dependencia de terceiro

O IP-Adapter exige o custom node `cubiq/ComfyUI_IPAdapter_plus` e dois pesos
auxiliares. A celula 6 pede **aceite explicito** antes de instalar. Se os
nodes nao aparecerem no servidor, o benchmark **PARA** — a Run 003 nunca cai
para uma referencia em silencio.


In [ ]:
#@title 0. Personagem, run e parametros { display-mode: "form" }
#@markdown Unico ponto a configurar. As celulas de execucao nao devem ser editadas.
CHARACTER_ID = "waifu_001"  #@param {type:"string"}

#@markdown ---
#@markdown **Qual run executar.** Todas sao IMG2IMG a partir de `full_body.png`.
#@markdown - **001/002** = img2img puro, sem IP-Adapter.
#@markdown - **003** = o mesmo img2img + 3 referencias via IP-Adapter.
RUN = "WAI - Run 001 (img2img de full_body, sem IP-Adapter)"  #@param ["WAI - Run 001 (img2img de full_body, sem IP-Adapter)", "WAI - Run 002 (repeticao exata da 001)", "WAI - Run 003 (img2img + IP-Adapter: full_body, face, outfit)"]

#@markdown ---
SEED = 42  #@param {type:"integer"}

#@markdown ---
#@markdown **denoise** — BASELINE_EXPERIMENTAL, nao otimizado.
#@markdown Precisa ser **< 1.0**: com 1.0 o latente de `full_body` e
#@markdown destruido e o processo vira txt2img.
#@markdown Mais baixo preserva a arte original mas resiste a virar chibi;
#@markdown mais alto vira chibi mas perde identidade.
DENOISE = 0.50  #@param {type:"slider", min:0.10, max:0.95, step:0.05}

#@markdown ---
#@markdown Pesos do IP-Adapter — **so valem na Run 003**.
#@markdown BASELINE_EXPERIMENTAL. Sem sweep nesta rodada.
WEIGHT_FULL_BODY = 1.0  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
WEIGHT_FACE = 0.6  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
WEIGHT_OUTFIT = 0.8  #@param {type:"slider", min:0.0, max:2.0, step:0.05}
COMBINE_METHOD = "concat"  #@param ["concat", "add", "average", "norm average", "subtract", "max", "min"]

RUN_ID = RUN.split("Run ")[1][:3]
IS_RUN_003 = RUN_ID == "003"
IS_BASELINE = not IS_RUN_003
MODEL_KEY = "wai_illustrious_sdxl_v170"
WORKFLOW = "experimental/wai_illustrious_ipadapter"
WORKFLOW_VERSION = "v2" if IS_RUN_003 else "v0"

# full_body e SEMPRE consumida: e a imagem de partida do img2img.
REFS_DECLARADAS = (["full_body", "face", "outfit"] if IS_RUN_003
                   else ["full_body"])
REFS_CONSUMIDAS = list(REFS_DECLARADAS)
PRIMARY_IMAGE_ROLE = "source_image"

assert DENOISE < 1.0, (
    "denoise TEM de ser < 1.0: com 1.0 o latente inicial de full_body e "
    "destruido e o img2img vira txt2img.")

print("personagem :", CHARACTER_ID)
print("run        :", RUN_ID)
print("workflow   :", f"{WORKFLOW}@{WORKFLOW_VERSION}")
print("seed       :", SEED, "| denoise:", DENOISE)
print()
print("PERSONAGEM NORMAL -> WAI -> CHIBI")
print("  imagem de partida : full_body.png (primary_image_role: source_image)")
print("  refs consumidas   :", REFS_CONSUMIDAS)
print()
if IS_BASELINE:
    print("Run", RUN_ID, "= img2img puro. Sem IP-Adapter, ControlNet, LoRA,")
    print("Hires ou ADetailer. So a imagem original guia o resultado.")
else:
    print("Run 003 = o MESMO img2img + IP-Adapter.")
    print("  pesos  : full_body", WEIGHT_FULL_BODY, "| face", WEIGHT_FACE,
          "| outfit", WEIGHT_OUTFIT)
    print("  combine:", COMBINE_METHOD, "(BASELINE_EXPERIMENTAL)")
    print()
    print("  full_body.png tem DOIS papeis nesta run:")
    print("    1. imagem inicial do img2img (VAEEncode)")
    print("    2. referencia do IP-Adapter (IPAdapterEncoder)")
    print("  Registrado como dual_role no recipe.")
print()
print("Comparacao 001 x 003 isola o efeito do IP-Adapter: o resto e igual.")


---

## Celula 1 — ambiente e repositorio

In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
ATUALIZAR_REPO = True  #@param {type:"boolean"}

import os, sys, subprocess, pathlib

%cd /content
if not pathlib.Path("/content/ChibiCreate/.git").exists():
    !git clone --branch $BRANCH $REPO_URL ChibiCreate
elif ATUALIZAR_REPO:
    !cd /content/ChibiCreate && git fetch origin $BRANCH && git checkout -B $BRANCH origin/$BRANCH

%cd /content/ChibiCreate
!git log --oneline -1

for _m in [k for k in list(sys.modules)
           if k.startswith("chibi") or k.startswith("scripts.chibi")]:
    del sys.modules[_m]
sys.path.insert(0, "/content/ChibiCreate")
sys.path.insert(0, "/content/ChibiCreate/scripts")

!pip install -q pyyaml pillow numpy


---

## Celula 2 — preflight (portao)

Se nao couber, **BLOCKED**. Sem fallback silencioso, sem CPU, sem trocar de
modelo, sem quantizar por conta propria.

In [ ]:
#@title 2. Preflight — GPU, VRAM, RAM, disco { display-mode: "form" }
import sys, json, shutil, subprocess, time

# Checkpoint SDXL fp16 (~8-10 GB) + IP-Adapter (~1 GB) + CLIP-Vision ViT-H
# (~2.5 GB no encode). Estimativa de registry, nao medicao nossa.
MIN_VRAM_GB = 12.0
MIN_DISK_GB = 20.0
MIN_RAM_GB  = 10.0

try:
    import torch
except ImportError:
    torch = None

print("=" * 62)
print("PREFLIGHT — detectado, nao presumido")
print("=" * 62)
print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__ if torch else "ausente")

if torch is None or not torch.cuda.is_available():
    print("CUDA   : INDISPONIVEL")
    print()
    print("=" * 62)
    print("BLOCKED — a sessao esta em CPU")
    print("=" * 62)
    print("Este notebook ja pede T4 por padrao (metadata accelerator=GPU,")
    print("gpuType=T4). O Colab, porem, ignora essa preferencia quando:")
    print("  - a copia aberta e antiga, salva antes desta correcao;")
    print("  - nao ha T4 disponivel na sua conta no momento;")
    print("  - a sessao foi reconectada como CPU apos expirar.")
    print()
    print("Corrija na sessao atual:")
    print("  Ambiente de execucao -> Alterar o tipo de ambiente de execucao")
    print("  -> Acelerador de hardware: GPU T4 -> Salvar")
    print()
    print("Se voce salvou uma copia no seu Drive, salve-a de novo DEPOIS de")
    print("trocar para T4: a preferencia fica gravada no arquivo .ipynb.")
    print()
    print("Nao ha fallback para CPU: SDXL em CPU levaria horas por imagem.")
    raise SystemExit("BLOCKED — sessao em CPU. Troque para GPU T4.")

props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024 ** 3
free_disk = shutil.disk_usage("/content").free / 1024 ** 3
try:
    import psutil
    ram = psutil.virtual_memory().total / 1024 ** 3
except ImportError:
    ram = float(subprocess.check_output(
        ["awk", "/MemTotal/ {print $2/1048576}", "/proc/meminfo"]).strip())

GPU_INFO = {
    "name": props.name, "vram_total_gb": round(vram, 2),
    "vram_free_gb": round(torch.cuda.mem_get_info()[0] / 1024 ** 3, 2),
    "cuda": torch.version.cuda,
    "capability": f"{props.major}.{props.minor}",
    "torch": torch.__version__, "python": sys.version.split()[0],
    "bf16_supported": props.major >= 8,
    "ram_gb": round(ram, 2), "disk_free_gb": round(free_disk, 2),
}
for k, v in GPU_INFO.items():
    print(f"  {k:18} {v}")

print()
falhas = []
if vram < MIN_VRAM_GB:
    falhas.append(f"VRAM {vram:.1f} < {MIN_VRAM_GB} GB")
if free_disk < MIN_DISK_GB:
    falhas.append(f"disco {free_disk:.1f} < {MIN_DISK_GB} GB")
if ram < MIN_RAM_GB:
    falhas.append(f"RAM {ram:.1f} < {MIN_RAM_GB} GB")

if not GPU_INFO["bf16_supported"]:
    print("AVISO: sem bf16 nativo (capability < 8.0, ex. T4). Cai para fp16.")

if falhas:
    print("=" * 62); print("BLOCKED"); print("=" * 62)
    for f in falhas:
        print(" -", f)
    print()
    print("Nao fazer fallback silencioso: nao trocar de modelo, nao remover")
    print("referencias, nao quantizar por conta propria. Reportar o bloqueio.")
    raise SystemExit("BLOCKED")

print("Hardware adequado.")
json.dump(GPU_INFO, open("/content/gpu_info_wai.json", "w"), indent=2)


---

## Celula 3 — entradas e hashes

As **mesmas** referencias do benchmark FLUX, derivadas de
`characters/<CHARACTER_ID>/reference/`. Os originais nunca sao modificados.

In [ ]:
#@title 3. Referencias — full_body, face, outfit { display-mode: "form" }
import hashlib, pathlib, json
import numpy as np
from PIL import Image

try:
    REFS_CONSUMIDAS, RUN_ID, CHARACTER_ID
except NameError:
    raise SystemExit("BLOCKED — execute a celula 0 antes desta.")

REF_DIR = pathlib.Path(
    f"/content/ChibiCreate/characters/{CHARACTER_ID}/reference")
ARQUIVOS = {"full_body": "full_body.png", "face": "face.png",
            "outfit": "outfit.png"}

# Papel de cada referencia nesta run. full_body e SEMPRE a imagem de
# partida do img2img; na Run 003 ela acumula o papel de referencia do
# IP-Adapter.
def _papeis(papel):
    if papel not in REFS_CONSUMIDAS:
        return []
    if papel == "full_body":
        return (["source_image", "ipadapter_reference"] if IS_RUN_003
                else ["source_image"])
    return ["ipadapter_reference"]

ENTRADAS = {}
for papel, arq in ARQUIVOS.items():
    p = REF_DIR / arq
    if not p.exists():
        print(f"  [ausente] {papel}: {p}")
        continue
    b = p.read_bytes()
    im = Image.open(p)
    ENTRADAS[papel] = {
        "file": arq, "path": str(p),
        "artifact_sha256": hashlib.sha256(b).hexdigest(),
        "pixel_sha256": hashlib.sha256(
            np.array(im.convert("RGBA")).tobytes()).hexdigest(),
        "size": list(im.size), "bytes": len(b),
        "usada_nesta_run": papel in REFS_CONSUMIDAS,
        "roles": _papeis(papel),
    }
    marca = ("+".join(_papeis(papel)) if papel in REFS_CONSUMIDAS
             else "nao usada nesta run")
    print(f"  {papel:10} {str(im.size):12} "
          f"{ENTRADAS[papel]['artifact_sha256'][:16]}  [{marca}]")

faltando = [r for r in REFS_CONSUMIDAS if r not in ENTRADAS]
if faltando:
    raise SystemExit(
        f"BLOCKED — a Run {RUN_ID} exige {REFS_CONSUMIDAS} e faltam "
        f"{faltando}. Nao executar com menos referencias em silencio.")

# A imagem de partida define a resolucao de saida: nao ha resize.
if "full_body" not in ENTRADAS:
    raise SystemExit(
        "BLOCKED — full_body.png e a imagem inicial do img2img em TODAS as "
        "runs. Sem ela nao ha o que converter em chibi.")
SRC_W, SRC_H = ENTRADAS["full_body"]["size"]

print()
print(f"Run {RUN_ID}: {len(REFS_CONSUMIDAS)} referencia(s) confirmada(s).")
print(f"Imagem de partida : full_body.png  {SRC_W}x{SRC_H}")
print(f"Resolucao de saida: {SRC_W}x{SRC_H} (herdada, sem resize — "
      "redimensionar deformaria a arte)")
if IS_RUN_003:
    print("full_body em papel DUPLO: latente inicial + referencia IP-Adapter.")
print("Arquivos NAO sao modificados — apenas lidos.")
json.dump(ENTRADAS, open("/content/wai_inputs.json", "w"), indent=2)


---

## Celula 4 — Google Drive: localizar e validar o checkpoint

O checkpoint ja existe no seu Drive. Esta celula **monta**, **localiza**,
**valida** e **liga** o arquivo ao ComfyUI — sem download, sem upload e sem
tocar no Civitai.

O arquivo original **nao e movido nem modificado**: tentamos symlink
primeiro (instantaneo) e so caimos para copia se o symlink nao funcionar
neste ambiente.

Sobre a versao: o arquivo e explicitamente `waiIllustriousSDXL_v170`. O
`modelVersionId` do Civitai fica `unknown/pending` se voce nao souber — isso
**nao** impede a execucao, porque a identidade real do arquivo e garantida
pelo **SHA256**, que e mais forte que um id de catalogo.


In [ ]:
#@title 4. Montar o Drive e validar o checkpoint { display-mode: "form" }
#@markdown Caminho DENTRO do seu Drive (sem `/content/drive/MyDrive/`).
CKPT_DRIVE_PATH = "ComfyUI_Data/models/checkpoints/waiIllustriousSDXL_v170.safetensors"  #@param {type:"string"}
#@markdown Metadados do Civitai. Opcionais: deixe vazio e ficam
#@markdown `unknown/pending` — o SHA256 e que identifica o arquivo.
CIVITAI_MODEL_ID = "827184"  #@param {type:"string"}
CIVITAI_VERSION_ID = ""  #@param {type:"string"}
LICENCA_EXIBIDA = "Commercial use allowed (conforme UI do Civitai)"  #@param {type:"string"}

import hashlib, json, os, pathlib, struct, time

from chibi import model_registry as mr

CFG = mr.get_model(MODEL_KEY)

# ---------------------------------------------------------------- 1. mount
DRIVE_ROOT = pathlib.Path("/content/drive")
if not (DRIVE_ROOT / "MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive ja montado.")

MYDRIVE = DRIVE_ROOT / "MyDrive"
if not MYDRIVE.exists():
    raise SystemExit("BLOCKED — Drive nao montado. Reexecute e autorize.")

# ------------------------------------------------------------- 2. localizar
origem = (MYDRIVE / CKPT_DRIVE_PATH).expanduser()
CAMINHO_LOGICO = f"My Drive/{CKPT_DRIVE_PATH}"

if not origem.exists():
    print("=" * 64)
    print("BLOCKED — checkpoint nao encontrado no caminho informado")
    print("=" * 64)
    print("procurado :", origem)
    print("logico    :", CAMINHO_LOGICO)
    print()
    pasta = origem.parent
    if pasta.exists():
        achados = sorted(p.name for p in pasta.glob("*.safetensors"))
        print(f"A pasta existe e contem {len(achados)} .safetensors:")
        for nome in achados[:25]:
            print("   -", nome)
        if not achados:
            print("   (nenhum)")
    else:
        print("A pasta nao existe:", pasta)
        anc = pasta
        while anc != MYDRIVE and not anc.exists():
            anc = anc.parent
        print("Ancestral existente:", anc)
        if anc.exists():
            subs = sorted(p.name for p in anc.iterdir() if p.is_dir())[:25]
            print("Subpastas:", subs or "(nenhuma)")
    print()
    print("Corrija CKPT_DRIVE_PATH no formulario acima e reexecute.")
    print("Nao vamos adivinhar nome de arquivo nem baixar do Civitai.")
    raise SystemExit("BLOCKED — informe o caminho correto do Drive.")

tamanho = origem.stat().st_size

# ----------------------------------------------- 3. validar antes de usar
def ler_header_safetensors(p):
    """Le o header JSON do safetensors sem carregar os pesos.

    Layout: 8 bytes little-endian com o tamanho do header, seguido do JSON.
    """
    with open(p, "rb") as f:
        (n,) = struct.unpack("<Q", f.read(8))
        if not (0 < n < 200_000_000):
            raise ValueError(f"tamanho de header implausivel: {n}")
        return json.loads(f.read(n).decode("utf-8"))

print()
print("Validando o arquivo (header, sem carregar os pesos)...")
try:
    header = ler_header_safetensors(origem)
except Exception as exc:
    raise SystemExit(
        f"BLOCKED — nao e um .safetensors valido: {type(exc).__name__}: {exc}")

chaves = [k for k in header if k != "__metadata__"]

# Assinaturas de arquitetura. O segundo text encoder (OpenCLIP bigG) e o
# que distingue SDXL de SD 1.5/2.x; o primeiro bloco do UNet confirma que
# ha um modelo de difusao aqui, e nao um LoRA ou um VAE solto.
tem_unet = any(k.startswith("model.diffusion_model.") for k in chaves)
tem_te1 = any(k.startswith("conditioner.embedders.0.") for k in chaves)
tem_te2 = any(k.startswith("conditioner.embedders.1.") for k in chaves)
tem_vae = any(k.startswith("first_stage_model.") for k in chaves)

print(f"  tensores               : {len(chaves)}")
print(f"  UNet                   : {'sim' if tem_unet else 'NAO'}")
print(f"  text encoder 1 (CLIP-L) : {'sim' if tem_te1 else 'NAO'}")
print(f"  text encoder 2 (bigG)   : {'sim' if tem_te2 else 'NAO'}  <- marca do SDXL")
print(f"  VAE                    : {'sim' if tem_vae else 'NAO'}")

problemas = []
if not tem_unet:
    problemas.append("sem UNet (model.diffusion_model.*) — nao e checkpoint")
if not tem_te2:
    problemas.append(
        "sem o 2o text encoder (conditioner.embedders.1.*) — nao e SDXL")
if tamanho < 3 * 1024 ** 3:
    problemas.append(
        f"tamanho {tamanho / 1024 ** 3:.2f} GB e pequeno demais para SDXL")

if problemas:
    print()
    print("=" * 64); print("BLOCKED — nao parece um checkpoint SDXL"); print("=" * 64)
    for x in problemas:
        print(" -", x)
    print()
    print("Nao vamos iniciar a inferencia com um arquivo invalido.")
    raise SystemExit("BLOCKED — checkpoint invalido")

meta_embutida = header.get("__metadata__", {}) or {}

# ----------------------------------------------------------- 4. SHA256
print()
print(f"Calculando SHA256 de {tamanho / 1024 ** 3:.2f} GB (leitura via Drive,")
print("pode levar alguns minutos)...")
t0 = time.time()
h = hashlib.sha256()
with open(origem, "rb") as f:
    for bloco in iter(lambda: f.read(1 << 23), b""):
        h.update(bloco)
CKPT_SHA256 = h.hexdigest()
print(f"  concluido em {time.time() - t0:.0f}s")

# --------------------------------------- 5. ligar ao ComfyUI (sem copiar)
CKPT_DIR = pathlib.Path("/content/ComfyUI/models/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
destino = CKPT_DIR / origem.name

metodo = None
if destino.is_symlink() or destino.exists():
    metodo = "ja_presente"
else:
    try:
        destino.symlink_to(origem)
        # Symlink so serve se der para LER de verdade atraves dele.
        with open(destino, "rb") as f:
            f.read(8)
        metodo = "symlink"
    except Exception as exc:
        print(f"  symlink indisponivel ({type(exc).__name__}); copiando...")
        if destino.is_symlink():
            destino.unlink()
        import shutil
        shutil.copy2(origem, destino)
        metodo = "copia"

# ------------------------------------------------------------- 6. relatorio
print()
print("=" * 64)
print("CHECKPOINT PRONTO")
print("=" * 64)
print(f"  caminho encontrado : {origem}")
print(f"  caminho logico     : {CAMINHO_LOGICO}")
print(f"  nome               : {origem.name}")
print(f"  tamanho            : {tamanho:,} bytes ({tamanho / 1024 ** 3:.2f} GB)")
print(f"  sha256             : {CKPT_SHA256}")
print(f"  origem             : google_drive")
print(f"  ligado ao ComfyUI  : {metodo} -> {destino}")
if meta_embutida:
    print(f"  metadata embutida  : {dict(list(meta_embutida.items())[:5])}")
print()
print("  Arquivo original do Drive NAO foi movido nem modificado.")
print("  Civitai NAO foi acessado.")

VERSAO = {
    "source": "google_drive",
    "drive_logical_path": CAMINHO_LOGICO,
    "drive_absolute_path": str(origem),
    "file": origem.name,
    "sha256": CKPT_SHA256,
    "size_bytes": tamanho,
    "link_method": metodo,
    "comfy_path": str(destino),
    "civitai_model_id": CIVITAI_MODEL_ID or "unknown/pending",
    "civitai_model_version_id": CIVITAI_VERSION_ID or "unknown/pending",
    "revision_from_filename": "v170",
    "license_displayed": LICENCA_EXIBIDA,
    "verified_by_agent": False,
    "sdxl_validated": True,
    "sdxl_validation": {
        "tensor_count": len(chaves), "has_unet": tem_unet,
        "has_text_encoder_1": tem_te1, "has_text_encoder_2": tem_te2,
        "has_vae": tem_vae,
    },
    "embedded_metadata": meta_embutida,
}

if VERSAO["civitai_model_version_id"] == "unknown/pending":
    print()
    print("[HUMAN REVIEW REQUIRED] modelVersionId: unknown/pending.")
    print("Nao bloqueia a execucao: o SHA256 acima identifica o arquivo de")
    print("forma mais forte que um id de catalogo. Voce pode preencher os")
    print("metadados depois — o hash ja esta gravado no recipe.")

json.dump(VERSAO, open("/content/wai_version.json", "w"), indent=2)


---

## Celula 5 — ComfyUI

In [ ]:
#@title 5. Subir o ComfyUI { display-mode: "form" }
import subprocess, time, urllib.request, json, pathlib, os, shutil

COMFY = pathlib.Path("/content/ComfyUI")
COMFY_REPO = "https://github.com/comfyanonymous/ComfyUI.git"

def comfy_no_ar(timeout=5):
    try:
        with urllib.request.urlopen(
                "http://127.0.0.1:8188/system_stats", timeout=timeout) as r:
            return json.load(r)
    except Exception:
        return None

# A celula 4 pode ter criado /content/ComfyUI/models/checkpoints para
# colocar o symlink do Drive. Entao "a pasta existe" NAO significa "o
# ComfyUI esta instalado" — o marcador real e o .git + o main.py.
instalado = (COMFY / ".git").is_dir() and (COMFY / "main.py").is_file()

if not instalado:
    if COMFY.exists() and any(COMFY.iterdir()):
        # git clone recusa diretorio nao vazio: clonamos ao lado e
        # mesclamos, preservando o que a celula 4 ja colocou.
        print("Pasta ja existe (checkpoint do Drive). Clonando e mesclando...")
        tmp = pathlib.Path("/content/_comfy_tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        subprocess.run(["git", "clone", "-q", COMFY_REPO, str(tmp)], check=True)
        for item in tmp.iterdir():
            destino = COMFY / item.name
            if not destino.exists():
                shutil.move(str(item), str(destino))
            elif item.is_dir():
                # models/ ja existe com o checkpoint: copia so o que falta.
                for sub in item.rglob("*"):
                    rel = sub.relative_to(item)
                    alvo = destino / rel
                    if sub.is_dir():
                        alvo.mkdir(parents=True, exist_ok=True)
                    elif not alvo.exists():
                        alvo.parent.mkdir(parents=True, exist_ok=True)
                        shutil.move(str(sub), str(alvo))
        shutil.rmtree(tmp, ignore_errors=True)
    else:
        subprocess.run(["git", "clone", "-q", COMFY_REPO, str(COMFY)], check=True)

    !pip install -q -r /content/ComfyUI/requirements.txt

if not (COMFY / ".git").is_dir():
    raise SystemExit(
        f"BLOCKED — {COMFY} nao e um clone do ComfyUI. Apague a pasta "
        "(preservando models/) e reexecute esta celula.")

COMFY_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=COMFY).decode().strip()
print("ComfyUI commit:", COMFY_COMMIT)

# O symlink do checkpoint tem de ter sobrevivido a mesclagem.
try:
    _ck = pathlib.Path(VERSAO["comfy_path"])
    print("checkpoint    :", _ck.name,
          "(ok)" if _ck.exists() else "AUSENTE — reexecute a celula 4")
except NameError:
    print("checkpoint    : celula 4 ainda nao executada")

# Marcador estavel para achar o processo depois. Sem isso o pkill teria
# de adivinhar a linha de comando: o processo sobe como "python main.py"
# (cwd=/content/ComfyUI), entao um pkill -f "ComfyUI/main.py" NAO casa.
COMFY_TAG = "chibi_wai_comfy"


def matar_comfy(verbose=True):
    """Derruba QUALQUER ComfyUI ouvindo na 8188 e so retorna quando cair.

    Necessario antes de carregar custom nodes novos: o ComfyUI le
    custom_nodes uma unica vez, no boot. Reaproveitar um servidor que
    subiu antes da instalacao faz os nodes novos simplesmente nao
    existirem — sem erro alguma, o que e o pior caso.
    """
    global PROC
    try:
        if PROC is not None:
            PROC.terminate()
            try:
                PROC.wait(timeout=30)
            except Exception:
                PROC.kill()
    except NameError:
        pass
    PROC = None

    # Mata por marcador, por padrao de comando e por porta — o processo
    # pode ter sobrevivido a um restart de kernel, sem Popen para nós.
    subprocess.run(["pkill", "-f", COMFY_TAG], check=False)
    subprocess.run(["pkill", "-f", "main.py --listen"], check=False)
    subprocess.run(["fuser", "-k", "8188/tcp"], check=False,
                   capture_output=True)

    for i in range(30):
        if comfy_no_ar(timeout=2) is None:
            if verbose:
                print(f"  servidor anterior derrubado (~{i}s)")
            return True
        time.sleep(2)
        if i == 14:
            subprocess.run(["pkill", "-9", "-f", "main.py --listen"],
                           check=False)
            subprocess.run(["fuser", "-k", "-9", "8188/tcp"], check=False,
                           capture_output=True)
    raise SystemExit(
        "BLOCKED — nao foi possivel derrubar o ComfyUI na porta 8188. "
        "Use Runtime > Restart session e execute de novo a partir da "
        "celula 4.")


def subir_comfy(force=False):
    """Sobe o ComfyUI. Com force=True, derruba o anterior antes.

    force e obrigatorio depois de instalar custom nodes.
    """
    global PROC
    if force:
        matar_comfy()
    elif comfy_no_ar():
        print("  ja estava no ar; reaproveitando.")
        PROC = None
        return None
    log = open("/content/comfyui_wai.log", "w")
    p = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=str(COMFY), stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, "CHIBI_COMFY_TAG": COMFY_TAG})
    for i in range(120):
        time.sleep(5)
        if comfy_no_ar():
            print(f"  no ar apos ~{(i + 1) * 5}s")
            PROC = p
            return p
        if p.poll() is not None:
            print(open("/content/comfyui_wai.log").read()[-3000:])
            raise SystemExit("ComfyUI morreu ao iniciar")
    raise SystemExit("ComfyUI nao respondeu em 10 min")


PROC = subir_comfy()
os.environ["CHIBI_COMFY_URL"] = "http://127.0.0.1:8188"


---

## Celula 6 — IP-Adapter (**pule nas Runs 001/002**)

O baseline nao usa IP-Adapter. Esta celula e a proxima **so sao necessarias
para a Run 003** — nas Runs 001/002 elas se autodesativam e nao instalam
nada.

| item | arquivo | licenca |
|---|---|---|
| custom node | `cubiq/ComfyUI_IPAdapter_plus` | Apache-2.0 |
| adapter | `ip-adapter-plus_sdxl_vit-h.safetensors` | Apache-2.0 |
| encoder | `CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors` | Apache-2.0 |


In [ ]:
#@title 6. Instalar IP-Adapter (so na Run 003) { display-mode: "form" }
#@markdown Necessario apenas para a Run 003. Ignorado nas Runs 001/002.
ACEITO_INSTALAR_IPADAPTER = False  #@param {type:"boolean"}

import pathlib, subprocess, hashlib, json, os

def sha256_of(caminho, chunk=1 << 20):
    """SHA256 lendo em blocos: os pesos nao cabem confortavelmente em RAM."""
    h = hashlib.sha256()
    with open(caminho, "rb") as fh:
        for bloco in iter(lambda: fh.read(chunk), b""):
            h.update(bloco)
    return h.hexdigest()

IPADAPTER_META = None

if IS_BASELINE:
    print("Run", RUN_ID, "= baseline puro: IP-Adapter NAO e usado.")
    print("Nada instalado. Siga para a proxima celula.")
else:
    IPA_DIR = pathlib.Path(
        "/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus")
    MODELS = pathlib.Path("/content/ComfyUI/models")
    CFG_IPA = CFG["ipadapter_models"]

    if not ACEITO_INSTALAR_IPADAPTER and not IPA_DIR.exists():
        print("=" * 62)
        print("BLOCKED — IP-Adapter nao instalado")
        print("=" * 62)
        print("custom node :", CFG["custom_node_repo"])
        print("nodes       :", ", ".join(CFG["custom_node_nodes"]))
        print()
        print("Sem ele nao ha multi-referencia. A Run 003 NAO roda com uma")
        print("referencia so — seria substituir multi-reference em silencio.")
        raise SystemExit("Marque ACEITO_INSTALAR_IPADAPTER para prosseguir.")

    if not IPA_DIR.exists():
        !git clone -q {CFG["custom_node_repo"]} {IPA_DIR}

    IPA_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=IPA_DIR).decode().strip()
    IPA_TAG = subprocess.run(["git", "describe", "--tags", "--always"],
                             cwd=IPA_DIR, capture_output=True,
                             text=True).stdout.strip()
    print("IP-Adapter node:", CFG["custom_node_repo"])
    print("  commit :", IPA_COMMIT)
    print("  tag    :", IPA_TAG)

    IPADAPTER_ASSETS = {}
    for papel, spec in CFG_IPA.items():
        destino = MODELS / spec["target_dir"].split("/", 1)[1] / spec["file"]
        destino.parent.mkdir(parents=True, exist_ok=True)
        if not destino.exists():
            url = (f"https://huggingface.co/{spec['repo']}/resolve/main/"
                   f"{spec['repo_path']}")
            print(f"  baixando {papel}: {spec['file']}")
            !wget -q --show-progress -O "{destino}" "{url}"
        if not destino.exists() or destino.stat().st_size < 1_000_000:
            raise SystemExit(
                f"BLOCKED — download de '{spec['file']}' falhou "
                f"({destino}). Nao seguir sem o peso.")
        real = sha256_of(destino)
        IPADAPTER_ASSETS[papel] = {
            "file": spec["file"], "repo": spec["repo"],
            "repo_path": spec["repo_path"], "license": spec["license"],
            "sha256": real, "size_bytes": destino.stat().st_size,
        }
        print(f"  {papel:11} {spec['file']}")
        print(f"    sha256   {real}")
        print(f"    tamanho  {destino.stat().st_size / 1e6:.0f} MB"
              f" | {spec['license']}")

    IPADAPTER_META = {
        "repo": CFG["custom_node_repo"],
        "commit": IPA_COMMIT, "revision": IPA_TAG,
        "nodes": CFG["custom_node_nodes"], "assets": IPADAPTER_ASSETS,
    }
    json.dump(IPADAPTER_META, open("/content/ipadapter_meta.json", "w"),
              indent=2)
    print()
    print("A proxima celula REINICIA o ComfyUI automaticamente: os\n"
          "custom nodes so sao lidos no boot do servidor.")


In [ ]:
#@title 7. Reiniciar e validar os nodes (so na Run 003) { display-mode: "form" }
import subprocess, time, urllib.request, json, pathlib, re

def _carregar_object_info():
    with urllib.request.urlopen(
            "http://127.0.0.1:8188/object_info", timeout=180) as r:
        return json.load(r)

if IS_BASELINE:
    print("Baseline: nenhum custom node a carregar.")
    if comfy_no_ar() is None:
        PROC = subir_comfy()
    OBJECT_INFO = _carregar_object_info()
    print("nodes no servidor:", len(OBJECT_INFO))
else:
    # RESTART OBRIGATORIO. O ComfyUI le custom_nodes so no boot: se o
    # servidor subiu na celula 5, antes do git clone da celula 6, os nodes
    # do IP-Adapter nao existem nele. Reaproveitar o processo antigo faz
    # os nodes "sumirem" sem nenhum erro.
    print("Reiniciando o ComfyUI para carregar os nodes do IP-Adapter...")
    PROC = subir_comfy(force=True)

    OBJECT_INFO = _carregar_object_info()
    print("nodes no servidor:", len(OBJECT_INFO))
    print()

    faltando = [n for n in CFG["custom_node_nodes"] if n not in OBJECT_INFO]
    for n in CFG["custom_node_nodes"]:
        print(f"  {'OK  ' if n in OBJECT_INFO else 'FALTA'} {n}")

    if faltando:
        print()
        print("=" * 62)
        print("BLOCKED — nodes IP-Adapter ausentes apos o restart")
        print("=" * 62)
        print("faltando:", faltando)
        print()

        IPA_DIR = pathlib.Path(
            "/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus")
        print("custom node no disco :", IPA_DIR.exists())
        if IPA_DIR.exists():
            print("  __init__.py        :", (IPA_DIR / "__init__.py").exists())
            print("  arquivos           :",
                  len(list(IPA_DIR.glob('*.py'))), "arquivos .py")
        ipa_models = pathlib.Path("/content/ComfyUI/models/ipadapter")
        cv_models = pathlib.Path("/content/ComfyUI/models/clip_vision")
        for d in (ipa_models, cv_models):
            print(f"  {d.name:18}:",
                  [f.name for f in d.glob('*')] if d.exists() else "AUSENTE")

        # A causa real quase sempre esta no log, numa linha de import.
        log = pathlib.Path("/content/comfyui_wai.log")
        if log.exists():
            texto = log.read_text(errors="replace")
            linhas = texto.split("\n")
            relevantes = [
                l for l in linhas
                if re.search(r"ipadapter|import fail|cannot import|"
                             r"ModuleNotFoundError|Traceback|error", l, re.I)]
            if relevantes:
                print()
                print("--- log do ComfyUI (linhas relevantes) ---")
                for l in relevantes[-40:]:
                    print("  ", l[:200])
        print()
        print("Se o log mostrar erro de import, e incompatibilidade entre o")
        print("custom node e a versao do ComfyUI. NAO substituir")
        print("multi-reference por uma imagem so.")
        raise SystemExit("BLOCKED — IP-Adapter indisponivel")

    print()
    print("IP-Adapter disponivel. Multi-referencia possivel.")
    print()
    # Os nomes existirem nao basta: os campos usados no grafo tambem
    # precisam existir, senao o erro so aparece na submissao.
    _enc = OBJECT_INFO["IPAdapterEncoder"]["input"]
    _campos = {**_enc.get("required", {}), **_enc.get("optional", {})}
    print("IPAdapterEncoder aceita:", ", ".join(sorted(_campos)))
    for campo in ("ipadapter", "image", "weight", "clip_vision"):
        if campo not in _campos:
            print(f"  [atencao] campo '{campo}' nao existe neste node — a"
                  " assinatura mudou; o grafo pode falhar na submissao.")


---

## Celula 8 — validar o grafo

Confere o workflow da run escolhida contra os nodes reais do servidor. Se
falhar, **pare** — nao edite o grafo para "fazer passar".

In [ ]:
#@title 8. Validar o workflow e mostrar o grafo resolvido { display-mode: "form" }
import json, hashlib, pathlib

WF_PATH = pathlib.Path(
    f"/content/ChibiCreate/workflows/{WORKFLOW}/{WORKFLOW_VERSION}.json")
WF_BYTES = WF_PATH.read_bytes()
WORKFLOW_SHA256 = hashlib.sha256(WF_BYTES).hexdigest()
wf = json.loads(WF_BYTES)
nodes = {k: v for k, v in wf.items() if not k.startswith("_")}
classes = {k: v["class_type"] for k, v in nodes.items()}

print("workflow :", f"{WORKFLOW}@{WORKFLOW_VERSION}")
print("sha256   :", WORKFLOW_SHA256)
print("nodes    :", len(nodes))
print()

# ---------------- checklist de validacao ----------------
print("=" * 62)
print("VALIDACAO ESTRUTURAL")
print("=" * 62)
falhas = []

def checa(cond, ok, erro):
    print(("  OK    " if cond else "  FALHA ") + (ok if cond else erro))
    if not cond:
        falhas.append(erro)

load_ids = [k for k, c in classes.items() if c == "LoadImage"]
enc_ids = [k for k, c in classes.items() if c == "VAEEncode"]
ks_id = next((k for k, c in classes.items() if c == "KSampler"), None)

checa(bool(load_ids), f"LoadImage existe ({len(load_ids)})", "LoadImage ausente")
checa(len(enc_ids) == 1, "VAEEncode existe", "VAEEncode ausente")
checa(ks_id is not None, "KSampler existe", "KSampler ausente")

# o latente inicial tem de vir de full_body, via VAEEncode
if ks_id and enc_ids:
    lat = nodes[ks_id]["inputs"]["latent_image"]
    checa(lat == [enc_ids[0], 0],
          "latente inicial vem do VAEEncode",
          f"latente vem de {lat}, nao do VAEEncode")
    px = nodes[enc_ids[0]]["inputs"]["pixels"]
    fb = px[0] if isinstance(px, list) else None
    checa(fb in load_ids and
          nodes[fb]["inputs"]["image"] == "%%REF_FULL_BODY%%",
          "VAEEncode recebe full_body.png",
          "VAEEncode nao recebe full_body")

checa(not any(c == "EmptyLatentImage" for c in classes.values()),
      "sem EmptyLatentImage (nao e txt2img)", "EmptyLatentImage presente")
checa(DENOISE < 1.0, f"denoise {DENOISE} < 1.0",
      f"denoise {DENOISE} destruiria o latente inicial")

ipa = [c for c in classes.values() if "IPAdapter" in c]
if IS_BASELINE:
    checa(not ipa, "Run 001/002 sem IP-Adapter", f"IP-Adapter no baseline: {ipa}")
    checa(len(load_ids) == 1, "1 imagem (full_body)", "numero de imagens errado")
else:
    n_enc_ipa = sum(1 for c in classes.values() if c == "IPAdapterEncoder")
    checa(n_enc_ipa == 3, "3 IPAdapterEncoder", f"{n_enc_ipa} encoders")
    checa(any(c == "IPAdapterCombineEmbeds" for c in classes.values()),
          "CombineEmbeds presente", "CombineEmbeds ausente")
    checa(len(load_ids) == 3, "3 imagens carregadas", "faltam imagens")
    # full_body precisa alimentar VAEEncode E um Encoder do IP-Adapter
    fb_id = next((k for k in load_ids
                  if nodes[k]["inputs"]["image"] == "%%REF_FULL_BODY%%"), None)
    usos = [k for k, v in nodes.items()
            if any(isinstance(x, list) and x[0] == fb_id
                   for x in v["inputs"].values())]
    tipos = {classes[u] for u in usos}
    checa({"VAEEncode", "IPAdapterEncoder"} <= tipos,
          "full_body em DOIS papeis (latente + referencia)",
          f"full_body usado so em {tipos}")
    ks_model = nodes[ks_id]["inputs"]["model"]
    checa(classes.get(ks_model[0]) == "IPAdapterEmbeds",
          "KSampler recebe o modelo com IP-Adapter aplicado",
          "KSampler nao usa a saida do IPAdapterEmbeds")

ausentes = [c for c in set(classes.values()) if c not in OBJECT_INFO]
checa(not ausentes, "todas as classes existem no servidor",
      f"classes ausentes: {ausentes}")

# nenhuma referencia declarada pode ficar de fora
placeholders = {v["inputs"]["image"] for k, v in nodes.items()
                if classes[k] == "LoadImage"}
esperados = {f"%%REF_{r.upper()}%%" for r in REFS_CONSUMIDAS}
checa(placeholders == esperados,
      f"referencias do grafo == declaradas ({sorted(REFS_CONSUMIDAS)})",
      f"grafo usa {placeholders}, esperado {esperados}")

if falhas:
    print()
    raise SystemExit(f"BLOCKED — validacao falhou: {falhas}")

print()
print("Validacao completa. Nenhuma referencia descartada em silencio.")


---

## Celula 9 — executar

Monta o grafo com os parametros e pesos declarados e envia ao ComfyUI.

In [ ]:
#@title 9. Montar o grafo, exibir e executar { display-mode: "form" }
import json, time, copy, hashlib, pathlib, urllib.request, uuid
import numpy as np
from PIL import Image

try:
    VERSAO
except NameError:
    raise SystemExit("BLOCKED — execute a celula 4 (Drive) antes desta.")
if VERSAO.get("source") != "google_drive" or not VERSAO.get("sha256"):
    raise SystemExit("BLOCKED — checkpoint nao validado. Reexecute a celula 4.")
if not VERSAO.get("sdxl_validated"):
    raise SystemExit("BLOCKED — checkpoint nao passou na validacao SDXL.")

CKPT_SHA256 = VERSAO["sha256"]
CKPT_PATH = pathlib.Path(VERSAO["comfy_path"])
if not CKPT_PATH.exists():
    raise SystemExit(
        f"BLOCKED — '{CKPT_PATH}' sumiu (sessao reiniciada?). "
        "Reexecute a celula 4.")

print("checkpoint:", CKPT_PATH.name, "|", VERSAO["link_method"])
print("sha256    :", CKPT_SHA256)

# Copia das referencias para o input do ComfyUI. Originais intocados.
COMFY_INPUT = pathlib.Path("/content/ComfyUI/input")
COMFY_INPUT.mkdir(parents=True, exist_ok=True)
nomes = {}
for papel in REFS_CONSUMIDAS:
    origem_ref = pathlib.Path(ENTRADAS[papel]["path"])
    nome = f"{CHARACTER_ID}_{papel}.png"
    (COMFY_INPUT / nome).write_bytes(origem_ref.read_bytes())
    nomes[papel] = nome

# SRC_W/SRC_H vem da celula 3 (imagem de partida). Nao ha resize.

PAR = CFG["parameters"]
PROMPT = " ".join(CFG["prompt_override"].split())
NEGATIVE = CFG["negative_prompt_override"]
PREFIX = f"wai_{CHARACTER_ID}_run{RUN_ID}_{uuid.uuid4().hex[:8]}"

SUBST = {
    "%%CKPT_NAME%%": CKPT_PATH.name,
    "%%PROMPT%%": PROMPT,
    "%%NEGATIVE_PROMPT%%": NEGATIVE,
    "%%REF_FULL_BODY%%": nomes.get("full_body"),
    "%%REF_FACE%%": nomes.get("face"),
    "%%REF_OUTFIT%%": nomes.get("outfit"),
    "%%WEIGHT_FULL_BODY%%": float(WEIGHT_FULL_BODY),
    "%%WEIGHT_FACE%%": float(WEIGHT_FACE),
    "%%WEIGHT_OUTFIT%%": float(WEIGHT_OUTFIT),
    "%%COMBINE_METHOD%%": COMBINE_METHOD,
    "%%WEIGHT_TYPE%%": "linear",
    "%%EMBEDS_SCALING%%": "V only",
    "%%IPADAPTER_FILE%%": CFG["ipadapter_models"]["adapter"]["file"],
    "%%CLIP_VISION_FILE%%": CFG["ipadapter_models"]["clip_vision"]["file"],
    "%%SEED%%": int(SEED), "%%STEPS%%": PAR["steps"], "%%CFG%%": PAR["cfg"],
    "%%SAMPLER%%": PAR["sampler"], "%%SCHEDULER%%": PAR["scheduler"],
    "%%DENOISE%%": float(DENOISE), "%%OUTPUT_PREFIX%%": PREFIX,
}

GRAFO = copy.deepcopy(nodes)
for nid, node in GRAFO.items():
    for campo, valor in node["inputs"].items():
        if isinstance(valor, str) and valor in SUBST:
            v = SUBST[valor]
            if v is None:
                raise SystemExit(f"BLOCKED — placeholder {valor} sem valor.")
            node["inputs"][campo] = v
    node.pop("_meta", None)

restantes = [v for n in GRAFO.values() for v in n["inputs"].values()
             if isinstance(v, str) and v.startswith("%%")]
assert not restantes, f"placeholders nao substituidos: {restantes}"

# ---------------- grafo resolvido, ANTES de executar ----------------
print()
print("=" * 62)
print("GRAFO RESOLVIDO (o que sera enviado ao ComfyUI)")
print("=" * 62)
_ordem = sorted(GRAFO, key=lambda k: int(k))
for nid in _ordem:
    n = GRAFO[nid]
    print(f"  [{nid}] {n['class_type']}")
    for campo, valor in n["inputs"].items():
        if isinstance(valor, list):
            print(f"        {campo:14} <- node[{valor[0]}]:{valor[1]}")
        else:
            txt = str(valor)
            print(f"        {campo:14} = "
                  f"{txt[:66] + '...' if len(txt) > 66 else txt}")
print()
print("FLUXO DO LATENTE")
_enc = next(k for k, v in GRAFO.items() if v["class_type"] == "VAEEncode")
_ks = next(k for k, v in GRAFO.items() if v["class_type"] == "KSampler")
_ld = GRAFO[_enc]["inputs"]["pixels"][0]
print(f"  {GRAFO[_ld]['inputs']['image']} -> LoadImage[{_ld}]"
      f" -> VAEEncode[{_enc}] -> KSampler[{_ks}] (denoise {DENOISE})")
print()
print("PARAMETROS EFETIVOS")
print(f"  resolucao : {SRC_W}x{SRC_H} (herdada da imagem de partida, sem resize)")
print(f"  steps/cfg : {PAR['steps']} / {PAR['cfg']}")
print(f"  sampler   : {PAR['sampler']} / {PAR['scheduler']}")
print(f"  denoise   : {DENOISE} (BASELINE_EXPERIMENTAL) | seed {SEED} | batch 1")
print(f"  hires     : {PAR['hires_fix']} | VAE: integrado ao checkpoint")
print()
print("PROMPT   :", PROMPT)
print("NEGATIVE :", NEGATIVE)
print()

t0 = time.time()
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=json.dumps({"prompt": GRAFO}).encode(),
    headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=60) as r:
        pid = json.load(r)["prompt_id"]
except urllib.error.HTTPError as e:
    detalhe = e.read().decode()[:3000]
    print(detalhe)
    pathlib.Path("/content/erro_wai.txt").write_text(
        f"etapa: submissao_do_grafo\n{detalhe}")
    raise SystemExit("BLOCKED na etapa 'submissao_do_grafo' — erro acima.")

print("prompt_id:", pid)
while True:
    time.sleep(3)
    with urllib.request.urlopen(
            f"http://127.0.0.1:8188/history/{pid}", timeout=30) as r:
        h = json.load(r)
    if pid in h:
        hist = h[pid]
        break
    if time.time() - t0 > 1800:
        raise SystemExit("BLOCKED na etapa 'timeout_execucao' — 30 min.")

EXEC_TIME = round(time.time() - t0, 2)
status = hist.get("status", {})
if status.get("status_str") == "error":
    detalhe = json.dumps(status, indent=2)[:3000]
    print(detalhe)
    pathlib.Path("/content/erro_wai.txt").write_text(
        f"etapa: execucao_do_grafo\n{detalhe}")
    raise SystemExit("BLOCKED na etapa 'execucao_do_grafo' — erro acima.")

saidas = [i for o in hist["outputs"].values() for i in o.get("images", [])]
assert saidas, "nenhuma imagem produzida"
info = saidas[0]
OUT = (pathlib.Path("/content/ComfyUI/output") /
       (info.get("subfolder") or "") / info["filename"])
print(f"gerado em {EXEC_TIME}s -> {OUT.name}")

fig_w = 12
_orig = Image.open(ENTRADAS["full_body"]["path"]).convert("RGB")
_novo = Image.open(OUT).convert("RGB")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(fig_w, fig_w / 2))
for a, (im, t) in zip(ax, [(_orig, "ENTRADA (full_body)"),
                           (_novo, f"SAIDA (denoise {DENOISE})")]):
    a.imshow(im); a.set_title(t, fontsize=11); a.axis("off")
plt.tight_layout(); plt.show()


---

## Celula 10 — recipe e hashes

Registro completo de reprodutibilidade. `artifact_sha256` (bytes do arquivo)
e `output_pixel_sha256` (conteudo dos pixels) sao gravados **separados**.

In [ ]:
#@title 10. Gravar recipe, workflow resolvido e logs { display-mode: "form" }
import json, hashlib, pathlib, shutil
import numpy as np
from PIL import Image

RUN_DIR = pathlib.Path(
    f"/content/ChibiCreate/experiments/model_eval/{MODEL_KEY}/run_{RUN_ID}")
if RUN_DIR.exists():
    n = 2
    while (alt := RUN_DIR.parent / f"run_{RUN_ID}_r{n}").exists():
        n += 1
    RUN_DIR = alt
    print(f"[no overwrite] run anterior preservada -> {RUN_DIR.name}")
RUN_DIR.mkdir(parents=True)

destino = RUN_DIR / "output.png"
shutil.copy2(OUT, destino)
img = Image.open(destino)

# Grafo COM os valores ja substituidos: e o que de fato rodou.
(RUN_DIR / "workflow.resolved.json").write_text(json.dumps(GRAFO, indent=2))

logs_dir = RUN_DIR / "logs"
logs_dir.mkdir()
log_src = pathlib.Path("/content/comfyui_wai.log")
if log_src.exists():
    shutil.copy2(log_src, logs_dir / "comfyui.log")
(logs_dir / "history.json").write_text(json.dumps(hist, indent=2, default=str))

RECIPE = {
    "benchmark": "wai_illustrious_sdxl_vs_flux2_klein_4b",
    "purpose": "comparative_benchmark_only",
    "is_official_pipeline": False,
    "run": RUN_ID,
    "run_type": ("img2img" if IS_BASELINE
                 else "img2img_plus_ipadapter_multiref"),
    "character": CHARACTER_ID,

    "model": {
        "key": MODEL_KEY, "file": VERSAO["file"], "sha256": CKPT_SHA256,
        "source": VERSAO["source"],
        "drive_logical_path": VERSAO["drive_logical_path"],
        "link_method": VERSAO["link_method"],
        "size_bytes": VERSAO["size_bytes"],
        "revision_from_filename": VERSAO["revision_from_filename"],
        "civitai_model_id": VERSAO["civitai_model_id"],
        "civitai_model_version_id": VERSAO["civitai_model_version_id"],
        "license": CFG["license_name"],
        "license_displayed": VERSAO["license_displayed"],
        "license_verified_by_agent": False,
        "commercial_status": CFG["commercial_status"],
        "sdxl_validated": VERSAO["sdxl_validated"],
        "sdxl_validation": VERSAO["sdxl_validation"],
        "embedded_metadata": VERSAO["embedded_metadata"],
    },

    "pipeline": "img2img",
    "primary_image": "full_body",
    "primary_image_role": PRIMARY_IMAGE_ROLE,
    "source_resolution": [SRC_W, SRC_H],
    "resolution_note": (
        "Saida herda a resolucao da imagem de partida. Nao ha resize: "
        "redimensionar deformaria a arte original."),
    "references_declared": REFS_DECLARADAS,
    "references_consumed": REFS_CONSUMIDAS,
    "reference_count": len(REFS_CONSUMIDAS),
    "reference_roles": (
        {"full_body": ["source_image"]} if IS_BASELINE else
        {"full_body": ["source_image", "ipadapter_reference"],
         "face": ["ipadapter_reference"],
         "outfit": ["ipadapter_reference"]}),
    "dual_role_note": (
        None if IS_BASELINE else
        "full_body.png tem DOIS papeis nesta run: imagem inicial do img2img "
        "(VAEEncode) e referencia do IP-Adapter (IPAdapterEncoder). "
        "Intencional e registrado."),
    "references_note": (
        "img2img puro: full_body e a imagem de partida (latente inicial). "
        "Nenhuma referencia descartada."
        if IS_BASELINE else
        "img2img a partir de full_body MAIS 3 referencias via IP-Adapter."),
    "references": {
        papel: {
            "file": ENTRADAS[papel]["file"],
            "artifact_sha256": ENTRADAS[papel]["artifact_sha256"],
            "pixel_sha256": ENTRADAS[papel]["pixel_sha256"],
            "weight": (SUBST[f"%%WEIGHT_{papel.upper()}%%"]
                       if IS_RUN_003 else None),
        } for papel in REFS_CONSUMIDAS
    },
    "reference_mechanism": None if IS_BASELINE else "ipadapter_encode_combine",
    "ipadapter": IPADAPTER_META,
    "weights_status": None if IS_BASELINE else "BASELINE_EXPERIMENTAL",
    "combine_method": None if IS_BASELINE else COMBINE_METHOD,

    "workflow": f"{WORKFLOW}@{WORKFLOW_VERSION}",
    "workflow_sha256": WORKFLOW_SHA256,
    "comfyui_commit": COMFY_COMMIT,
    "custom_nodes": ([] if IS_BASELINE
                     else [{"repo": IPADAPTER_META["repo"],
                            "commit": IPADAPTER_META["commit"],
                            "revision": IPADAPTER_META["revision"]}]),

    "prompt": PROMPT,
    "negative_prompt": NEGATIVE,
    "prompt_mode": CFG["prompt_mode"],
    "parameters": {
        "seed": int(SEED), "steps": PAR["steps"], "cfg": PAR["cfg"],
        "sampler": PAR["sampler"], "scheduler": PAR["scheduler"],
        "denoise": float(DENOISE),
        "denoise_status": "BASELINE_EXPERIMENTAL",
        "resolution": [SRC_W, SRC_H],
        "batch": 1, "hires_fix": PAR["hires_fix"],
        "vae": "integrado ao checkpoint",
    },
    "parameters_source": " ".join(PAR["parameters_source"].split()),

    "hardware": json.load(open("/content/gpu_info_wai.json")),
    "execution_time_s": EXEC_TIME,
    "artifact_sha256": hashlib.sha256(destino.read_bytes()).hexdigest(),
    "output_pixel_sha256": hashlib.sha256(
        np.array(img.convert("RGBA")).tobytes()).hexdigest(),
    "output_size": list(img.size),
    "determinism_note": (
        "Hash identico entre runs NAO e garantido. Nao prometemos "
        "determinismo absoluto entre execucoes."),
}
(RUN_DIR / "recipe.json").write_text(json.dumps(RECIPE, indent=2, default=str))

print("run dir :", RUN_DIR)
for f in sorted(RUN_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(RUN_DIR)}  ({f.stat().st_size:,} bytes)")
print()
for k in ("artifact_sha256", "output_pixel_sha256", "workflow_sha256",
          "execution_time_s", "reference_count", "output_size"):
    print(f"  {k:22} {RECIPE[k]}")


---

## Celula 11 — reprodutibilidade (001 vs 002)

Rode depois de ter executado as Runs 001 e 002.

In [ ]:
#@title 11. Diagnostico e reprodutibilidade { display-mode: "form" }
import json, pathlib

base = pathlib.Path(
    f"/content/ChibiCreate/experiments/model_eval/{MODEL_KEY}")
runs = {p.name: json.load(open(p / "recipe.json"))
        for p in sorted(base.glob("run_*")) if (p / "recipe.json").exists()}
print("runs registradas:", list(runs))

a, b = runs.get("run_001"), runs.get("run_002")
if a and b:
    print()
    print("REPRODUTIBILIDADE (001 vs 002)")
    for campo in ("prompt", "negative_prompt", "parameters",
                  "references_consumed", "workflow_sha256"):
        igual = a[campo] == b[campo]
        print(f"  {'OK   ' if igual else 'DIFERE'} {campo}")
    assert a["model"]["sha256"] == b["model"]["sha256"], "checkpoint diferente"
    IDENTICO_BYTE = a["artifact_sha256"] == b["artifact_sha256"]
    IDENTICO_PIXEL = a["output_pixel_sha256"] == b["output_pixel_sha256"]
    print()
    print(f"  artifact_sha256 (byte)  "
          f"{'IDENTICO' if IDENTICO_BYTE else 'DIFERENTE'}")
    print(f"    001: {a['artifact_sha256']}")
    print(f"    002: {b['artifact_sha256']}")
    print(f"  output_pixel_sha256     "
          f"{'IDENTICO' if IDENTICO_PIXEL else 'DIFERENTE'}")
    print(f"    001: {a['output_pixel_sha256']}")
    print(f"    002: {b['output_pixel_sha256']}")
    print()
    if IDENTICO_PIXEL:
        print("  Reproducao deterministica NESTE ambiente, nesta sessao.")
        print("  Isso NAO garante o mesmo hash em outra GPU ou outra versao")
        print("  de torch/ComfyUI. Nao prometemos determinismo absoluto.")
    else:
        print("  Pixels DIFEREM com parametros identicos. Nao e falha do")
        print("  benchmark: e o resultado da medicao. Causas tipicas sao")
        print("  nao-determinismo de kernels CUDA e ordem de reducao em")
        print("  float. Fica registrado como medido.")
    DETERMINISMO = {
        "byte_identical": IDENTICO_BYTE, "pixel_identical": IDENTICO_PIXEL,
        "scope": "mesma sessao, mesma GPU, mesmas versoes",
    }
else:
    DETERMINISMO = None
    print()
    print("Execute as Runs 001 e 002 para a checagem de reprodutibilidade.")

print()
print("=" * 62)
print("DIAGNOSTICO — separar as tres causas possiveis")
print("=" * 62)
print("Olhe a Run 001 (img2img de full_body, sem IP-Adapter) e responda:")
print()
print("  A imagem esta TECNICAMENTE LIMPA?")
print("  (sem artefato cromatico, sem deformacao, anatomia coerente)")
print()
print("  NAO -> a causa e (A) configuracao/checkpoint.")
print("         O IP-Adapter esta INOCENTE: ele nem participou desta run.")
print("         PARE e reporte. Nao rode a 003 ainda.")
print()
print("  SIM -> o checkpoint funciona. Rode a Run 003 e compare:")
print("         003 limpa   -> causa (C): WAI funciona, resta julgar se e")
print("                        artisticamente melhor que o FLUX.")
print("         003 corrompida -> causa (B): WAI + IP-Adapter. Investigar")
print("                        ipadapter model, CLIP Vision, combinacao de")
print("                        embeddings, pesos, versoes de node,")
print("                        conditioning. UM fator por vez.")
print()
print("[HUMAN REVIEW REQUIRED] — a leitura tecnica e sua.")


---

## Celula 12 — montagem comparativa

Sem ranking automatico. A leitura e humana.

In [ ]:
#@title 12. Comparacao visual { display-mode: "form" }
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = pathlib.Path("/content/ChibiCreate")
BASE = ROOT / f"experiments/model_eval/{MODEL_KEY}"

def _abrir(p):
    p = pathlib.Path(p) if p else None
    return Image.open(p).convert("RGB") if p and p.exists() else None

paineis = [
    (_abrir(ROOT / f"characters/{CHARACTER_ID}/reference/full_body.png"),
     "ORIGINAL"),
    (_abrir(ROOT / f"characters/{CHARACTER_ID}/chibi/run_003_output.png"),
     "FLUX RUN 003\n(3 refs, ReferenceLatent)"),
    (_abrir(BASE / "run_001/output.png"),
     "WAI RUN 001\n(img2img, sem IP-Adapter)"),
    (_abrir(BASE / "run_002/output.png"), "WAI RUN 002\n(repeticao)"),
    (_abrir(BASE / "run_003/output.png"), "WAI RUN 003\n(3 refs, IP-Adapter)"),
]
paineis = [(im, t) for im, t in paineis if im is not None]

n = len(paineis)
fig, ax = plt.subplots(1, n, figsize=(4.4 * n, 6.8))
for a, (im, t) in zip(np.atleast_1d(ax), paineis):
    a.imshow(im); a.set_title(t, fontsize=10); a.axis("off")
plt.tight_layout()
COMPARISON = pathlib.Path("/content/comparison.png")
plt.savefig(COMPARISON, dpi=110, bbox_inches="tight")
plt.show()
print("comparacao salva em:", COMPARISON)

print()
print("=" * 62)
print("[HUMAN REVIEW REQUIRED] — sem ranking automatico")
print("=" * 62)
print("PRIMEIRO: a imagem esta tecnicamente limpa?")
for item in ["sem artefato cromatico", "sem deformacao",
             "anatomia coerente", "sem ruido/banding"]:
    print("  [ ]", item)
print()
print("DEPOIS: STYLE / IDENTITY / DESIGN_PRESERVATION")
print("Eixo principal: DESIGN_PRESERVATION")
for item in ["roupa", "capa", "ornamentos dourados", "acessorios", "cabelo",
             "chifres", "silhueta", "proporcoes chibi",
             "fidelidade das cores"]:
    print("  [ ]", item)
print()
print("Como ler cada painel:")
print("  - WAI 001/002: img2img de full_body, denoise 0.50, SEM IP-Adapter.")
print("  - WAI 003    : o MESMO img2img + 3 refs via IP-Adapter.")
print("  - 001 x 003 isola o efeito do IP-Adapter: o resto e identico.")
print()
print("Assimetria registrada em relacao ao FLUX:")
print("  - Os prompts diferem (o autor do WAI exige prompt curto).")
print("  - Resolucao coincide: ambos 1024x1024.")


In [ ]:
#@title 13. Empacotar tudo em ZIP { display-mode: "form" }
import json, hashlib, pathlib, shutil, zipfile, datetime

ROOT = pathlib.Path("/content/ChibiCreate")
BASE = ROOT / f"experiments/model_eval/{MODEL_KEY}"
STAGE = pathlib.Path("/content/_wai_zip")
if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir()

runs = sorted(p for p in BASE.glob("run_*") if (p / "recipe.json").exists())
if not runs:
    raise SystemExit("BLOCKED — nenhuma run registrada. Execute antes.")

ESPERADAS = {"run_001", "run_002", "run_003"}
PRESENTES = {r.name for r in runs}
FALTANDO = sorted(ESPERADAS - PRESENTES)
if FALTANDO:
    print("=" * 62)
    print("ATENCAO — ZIP INCOMPLETO")
    print("=" * 62)
    print("faltam:", FALTANDO)
    print("Volte a celula 0, troque RUN e execute as celulas 8->10 para")
    print("cada run que falta. So depois gere o ZIP final.")
    print("O ZIP sera gerado assim mesmo, marcado como PARCIAL.")
    print()

# 1. runs completas (output, recipe, workflow resolvido, logs)
for r in runs:
    shutil.copytree(r, STAGE / r.name)

# 2. comparacao
comp = pathlib.Path("/content/comparison.png")
if comp.exists():
    shutil.copy2(comp, STAGE / "comparison.png")
else:
    print("[aviso] comparison.png ausente — rode a celula 12 antes.")

# 3. hashes de tudo que entrou no pacote
hashes = {}
for f in sorted(STAGE.rglob("*")):
    if f.is_file():
        hashes[str(f.relative_to(STAGE))] = {
            "sha256": hashlib.sha256(f.read_bytes()).hexdigest(),
            "bytes": f.stat().st_size,
        }
(STAGE / "hashes.json").write_text(json.dumps(hashes, indent=2))

# 4. relatorio final
recipes = {r.name: json.load(open(r / "recipe.json")) for r in runs}
qualquer = next(iter(recipes.values()))
L = []
L.append("# WAI-illustrious-SDXL — resultados do benchmark comparativo\n")
if FALTANDO:
    L.append(f"> **PACOTE PARCIAL** — faltam: {', '.join(FALTANDO)}. "
             "Execute-as antes de qualquer comparacao.\n")
L.append(f"Gerado em {datetime.datetime.now(datetime.timezone.utc).isoformat()}")
L.append(f"Personagem: {qualquer['character']}\n")
L.append("> BENCHMARK COMPARATIVO. Nao e pipeline oficial. O benchmark FLUX")
L.append("> nao foi modificado nem reexecutado.\n")

L.append("## Checkpoint\n")
m = qualquer["model"]
L.append(f"- arquivo: `{m['file']}`")
L.append(f"- sha256: `{m['sha256']}`")
L.append(f"- origem: {m['source']} (`{m['drive_logical_path']}`)")
L.append(f"- tamanho: {m['size_bytes']:,} bytes")
L.append(f"- model id: {m['civitai_model_id']}")
L.append(f"- modelVersionId: {m['civitai_model_version_id']}")
L.append(f"- versao (nome do arquivo): {m['revision_from_filename']}")
L.append(f"- licenca: {m['license']} — {m['commercial_status']}")
L.append(f"- validado como SDXL: {m['sdxl_validated']}\n")

L.append("## Runs\n")
L.append("| run | tipo | refs consumidas | resolucao | tempo | pixel sha256 |")
L.append("|---|---|---|---|---|---|")
for nome, rec in recipes.items():
    L.append(
        f"| {nome} | {rec['run_type']} | "
        f"{rec['references_consumed'] or '(nenhuma)'} | "
        f"{rec['output_size'][0]}x{rec['output_size'][1]} | "
        f"{rec['execution_time_s']}s | `{rec['output_pixel_sha256'][:16]}...` |")

p_ = qualquer["parameters"]
L.append("\n## Parametros\n")
for k, v in p_.items():
    L.append(f"- {k}: `{v}`")
L.append(f"\nFonte: {qualquer['parameters_source']}\n")
L.append(f"**Prompt** (`{qualquer['prompt_mode']}`):\n\n> {qualquer['prompt']}\n")
L.append(f"**Negativo**:\n\n> {qualquer['negative_prompt']}\n")

L.append("## Ambiente\n")
hw = qualquer["hardware"]
for k in ("name", "vram_total_gb", "ram_gb", "cuda", "torch", "python"):
    L.append(f"- {k}: {hw.get(k)}")
L.append(f"- ComfyUI: `{qualquer['comfyui_commit']}`")
# Custom nodes de TODAS as runs: usar so a primeira esconderia os nodes
# da Run 003, que e justamente quem os usa.
_cns = {(cn["repo"], cn["commit"], cn["revision"])
        for rec in recipes.values() for cn in (rec.get("custom_nodes") or [])}
for repo, commit, rev in sorted(_cns):
    L.append(f"- custom node: {repo} @ `{commit}` ({rev})")
L.append("- Runs 001/002: nenhum custom node (apenas nodes Core do ComfyUI)")
if not _cns:
    L.append("- Run 003 nao executada: nenhum custom node registrado")

L.append("\n## Pipeline\n")
L.append("**As tres runs sao img2img real.** O latente inicial vem sempre de")
L.append("`full_body.png`:\n")
L.append("```")
L.append("full_body.png -> LoadImage -> VAEEncode -> KSampler(latent_image)")
L.append("              -> VAEDecode -> output.png")
L.append("```\n")
L.append("- **Run 001** — img2img de `full_body.png`, denoise "
         f"{p_['denoise']}, SEM IP-Adapter.")
L.append("- **Run 002** — reproducao exata da Run 001 (mesmo checkpoint,")
L.append("  workflow, imagem, parametros e seed).")
L.append("- **Run 003** — o MESMO img2img + multiplas referencias via")
L.append("  IP-Adapter. `full_body.png` acumula dois papeis: latente")
L.append("  inicial (`VAEEncode`) e referencia (`IPAdapterEncoder`).\n")
L.append("Como o ramo do latente e identico, comparar 001 x 003 isola")
L.append("exatamente o efeito do IP-Adapter.\n")

L.append("## ERRO CORRIGIDO\n")
L.append("Uma versao anterior deste relatorio afirmava que as Runs 001/002")
L.append("eram **txt2img puro** e que `full_body` era declarada mas nao")
L.append("consumida. **Isso estava errado.**\n")
L.append("A causa do erro foi confundir *ausencia de IP-Adapter* com")
L.append("*ausencia de imagem inicial*. Sao coisas diferentes:\n")
L.append("- **IP-Adapter NAO e necessario para img2img.** Ele injeta")
L.append("  referencia visual no condicionamento, via CLIP Vision — e um")
L.append("  caminho paralelo, opcional.")
L.append("- **`VAEEncode` ja faz a imagem entrar no latente do KSampler.**")
L.append("  E node Core do ComfyUI e nao depende de custom node algum.")
L.append("- **Portanto a Run 001 e img2img mesmo sem IP-Adapter.** O que a")
L.append("  ausencia do IP-Adapter significa e que a imagem guia a geracao")
L.append("  apenas pelo latente inicial, nao pelo condicionamento.\n")
L.append("Consequencia: e **incorreto** dizer que a Run 001 mede apenas o")
L.append("checkpoint. Ela mede a conversao da personagem real em chibi via")
L.append("img2img — que e o objetivo do benchmark.\n")

if DETERMINISMO is not None:
    L.append("## Reprodutibilidade (Run 001 vs Run 002)\n")
    r1, r2 = recipes["run_001"], recipes["run_002"]
    L.append(f"- byte (artifact_sha256): "
             f"**{'IDENTICO' if DETERMINISMO['byte_identical'] else 'DIFERENTE'}**")
    L.append(f"  - run_001: `{r1['artifact_sha256']}`")
    L.append(f"  - run_002: `{r2['artifact_sha256']}`")
    L.append(f"- pixel (output_pixel_sha256): "
             f"**{'IDENTICO' if DETERMINISMO['pixel_identical'] else 'DIFERENTE'}**")
    L.append(f"  - run_001: `{r1['output_pixel_sha256']}`")
    L.append(f"  - run_002: `{r2['output_pixel_sha256']}`")
    L.append(f"- escopo da afirmacao: {DETERMINISMO['scope']}\n")
    if DETERMINISMO["pixel_identical"]:
        L.append("Reproducao deterministica **neste ambiente**. Isso nao")
        L.append("garante o mesmo hash em outra GPU ou outra versao de")
        L.append("torch/ComfyUI: nao afirmamos determinismo absoluto.\n")
    else:
        L.append("Os pixels diferem apesar de parametros identicos. Nao e")
        L.append("falha: e o resultado medido. Causa tipica e")
        L.append("nao-determinismo de kernels CUDA.\n")
else:
    L.append("## Reprodutibilidade\n")
    L.append("[PENDENTE] Runs 001 e 002 nao estao ambas presentes.\n")

r3 = recipes.get("run_003")
if r3:
    L.append("## Run 003 — multi-referencia\n")
    L.append("| referencia | arquivo | peso | papeis | sha256 |")
    L.append("|---|---|---|---|---|")
    for papel, ref in r3["references"].items():
        L.append(f"| {papel} | `{ref['file']}` | {ref.get('weight')} | "
                 f"{'+'.join(r3['reference_roles'].get(papel, []))} | "
                 f"`{ref['artifact_sha256'][:16]}...` |")
    L.append(f"\n- combine method: `{r3['combine_method']}`")
    L.append(f"- pesos: {r3['weights_status']}")
    L.append(f"- mecanismo: `{r3['reference_mechanism']}`")
    for papel, a_ in ((r3.get("ipadapter") or {}).get("assets") or {}).items():
        L.append(f"- {papel}: `{a_['file']}` ({a_['repo']}) — "
                 f"sha256 `{a_['sha256']}` — {a_['license']}")
    if r3.get("dual_role_note"):
        L.append(f"\n{r3['dual_role_note']}\n")

L.append("\n## Leitura\n")
L.append("- Run 001 tecnicamente suja -> causa (A) configuracao/checkpoint.")
L.append("  O IP-Adapter esta inocente: nao participou desta run.")
L.append("- Run 001 limpa + 003 suja -> causa (B) WAI + IP-Adapter.")
L.append("- Ambas limpas -> causa (C) julgamento artistico vs FLUX.\n")
L.append("[HUMAN REVIEW REQUIRED] Avaliacao artistica e humana. Sem")
L.append("ranking automatico e sem conclusao artistica nesta etapa.")
(STAGE / "RELATORIO.md").write_text("\n".join(L))

# 5. zip
ZIP_PATH = pathlib.Path("/content/wai_illustrious_eval_results.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(STAGE.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(STAGE))

print("=" * 62)
print("ZIP PRONTO")
print("=" * 62)
print("caminho :", ZIP_PATH)
print("tamanho :", f"{ZIP_PATH.stat().st_size:,} bytes")
print()
with zipfile.ZipFile(ZIP_PATH) as z:
    for n in sorted(z.namelist()):
        print("  ", n)

# Download automatico; se o navegador bloquear, o caminho acima serve.
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print()
    print("Download iniciado. Se o navegador bloquear, baixe pelo painel")
    print("de arquivos a esquerda:", ZIP_PATH)
except Exception as exc:
    print()
    print(f"[aviso] download automatico indisponivel ({type(exc).__name__}).")
    print("Baixe pelo painel de arquivos a esquerda:", ZIP_PATH)


---

## PARE AQUI

Concluido: Runs 001, 002, 003, receitas, hashes e o ZIP.

O objetivo desta rodada era **separar tres causas**, nao escolher vencedor:

| observacao | causa | proximo passo |
|---|---|---|
| Run 001 ja suja | **(A)** configuracao / checkpoint | parar e reportar — IP-Adapter nao participou |
| 001 limpa, 003 suja | **(B)** WAI + IP-Adapter | investigar **um** fator por vez |
| ambas limpas | **(C)** questao artistica | comparar com o FLUX |

### [HUMAN REVIEW REQUIRED]

- A imagem esta tecnicamente limpa?
- Concluir com **uma** marca: `PROMISING` / `INSUFFICIENT` / `BLOCKED`.

**Nao** fazer agora: calibrar pesos, Hires fix, outros checkpoints, Pony,
outro IP-Adapter, inpainting, design transfer, ou qualquer mudanca no FLUX.
